In [105]:
from agent_framework import (
    Executor,
    WorkflowContext,
    handler,
    executor,
    WorkflowBuilder,
)
import asyncio

## Signature / Outputs Guide

`ctx: WorkflowContext[str]`: used with send_message to send a [str] message to the next executor

`ctx: WorkflowContext[Never, str]`: used with ctx.yield_output to send a [str] workflow output to the caller

`ctx: WorkflowContext[str, str]`: send_message[str] and yield_output[str]


Explicitly define type parameters:

`@handler(input=str | int, output=str, workflow_output=bool)`: 

Implicitly define type parameters (will be inferred through introspection)

```
@handler()
async def my_method(self, input: str | int, ctx: WorkflowContext[str, bool])
``` 



In [ ]:
class UpperCase(Executor):
    def __init__(self, id: str = "upper_case_executor") -> None:
        super().__init__(id=id)
    @handler
    async def to_upper_case(self, text: str, ctx: WorkflowContext[str]) -> None:
        """Convert the input to uppercase and forward it to the next node."""
        await ctx.send_message(text.upper())

    # Note: this handle will always be called for a str type because handlers are registered in reverse order, so the most recently added handler that matches the input type will be used.
    @handler
    async def to_lower_case(self, text: str | int, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(str(text)[:3].lower())

    @handler
    async def to_double_integer(self, number: int, ctx: WorkflowContext[int]) -> None:
        await ctx.send_message(number * 2)

In [107]:
@executor()
async def reverse_text(text: str, ctx: WorkflowContext[str, int]) -> None:
    """Reverse the input text and forward it to the next node."""
    await ctx.send_message(text[::-1])

In [108]:
upper_case_executor = UpperCase()
upper_case_executor.type

'UpperCase'

In [109]:
upper_case_executor.to_json()

'{"id": "upper_case_executor", "type": "UpperCase"}'

In [110]:
reverse_text.to_json()

'{"id": "reverse_text", "type": "FunctionExecutor"}'

In [111]:
reverse_text.input_types

[str]

In [112]:
reverse_text.output_types

[str]

In [113]:
reverse_text.type

'FunctionExecutor'

In [ ]:
# Note: I can get the FunctionExecutor class if it is from a function if that helps


# Note: class based executors can have multiple handlers
# Note: function based executors have a single handler, which is the function itself
# Note: class based executors require an id to be provided, function based executors use the function name as the id if not provided
# Note: handlers cannot have the same signature. If two handlers have union types overlap, handlers are registered in reverse order, so the most recently added handler that matches the input type will be used.
# Note: ^MAF does not have documentation for this behaviour, yet but I have tested to confirm it. Handlers are executed in the order they are returned from the executor's _handler_specs property, which is in reverse order of registration.

def get_executor_metadata(executor: Executor) -> dict:
    return {
        "id": executor.id,
        "type": executor.type,
        "handlers": executor._handler_specs,
    }

get_executor_metadata(upper_case_executor)
# get_executor_metadata(reverse_text)

{'id': 'upper_case_executor',
 'type': 'UpperCase',
 'handlers': [{'name': 'to_double_integer',
   'message_type': int,
   'output_types': [int],
   'workflow_output_types': [],
   'ctx_annotation': agent_framework._workflows._workflow_context.WorkflowContext[int, typing.Never]},
  {'name': 'to_lower_case',
   'message_type': str | int,
   'output_types': [str],
   'workflow_output_types': [],
   'ctx_annotation': agent_framework._workflows._workflow_context.WorkflowContext[str, typing.Never]},
  {'name': 'to_upper_case',
   'message_type': str,
   'output_types': [str],
   'workflow_output_types': [],
   'ctx_annotation': agent_framework._workflows._workflow_context.WorkflowContext[str, typing.Never]}]}

In [115]:
workflow = WorkflowBuilder(start_executor=upper_case_executor).add_edge(upper_case_executor, reverse_text).build()
result = await workflow.run("hello there") # Note: notebook runner already has an event loop running
print(result)

[WorkflowEvent(type='executor_invoked', executor_id='upper_case_executor', data='hello there'), WorkflowEvent(type='executor_completed', executor_id='upper_case_executor', data=['hel']), WorkflowEvent(type='superstep_started', iteration=1), WorkflowEvent(type='executor_invoked', executor_id='reverse_text', data='hel'), WorkflowEvent(type='executor_completed', executor_id='reverse_text', data=['leh']), WorkflowEvent(type='superstep_completed', iteration=1), WorkflowEvent(type='superstep_started', iteration=2), WorkflowEvent(type='superstep_completed', iteration=2)]
